# PyDMA: Getting Started Tutorial

This notebook demonstrates how to use PyDMA (Python Degradation Mode Analysis) for analyzing battery degradation mechanisms using **real measurement data**.

## Overview

PyDMA quantifies three main degradation mechanisms:
- **LLI**: Loss of Lithium Inventory (charge carrier loss)
- **LAM_an**: Loss of Active Material at Anode  
- **LAM_ca**: Loss of Active Material at Cathode

The analysis works by fitting half-cell OCP curves to measured full-cell pseudo-OCV data.
**Notes:**
- **Blend electrodes are supported for both anode and cathode** (e.g., Si-Gr anode or blended cathode).
- **Inhomogeneity modeling is optional** and can be enabled per electrode (see config flags in the next section).


**Data used in this tutorial:**
- **Cathode**: NCA (P45B) - GITT pOCP (.csv)
- **Anode**: Graphite (Kuecher) - Lithiation curve
- **Full-cell**: P45B pOCV charging data


## 1. Installation

Install PyDMA using pip:

In [ ]:
# Install PyDMA (uncomment if needed)
# !pip install pydma

## 2. Import Required Modules

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Match notebook figure typography with PyDMA MATLAB-style balancing plots
plt.rcParams.update(
    {
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 12,
        "legend.fontsize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "lines.linewidth": 1.5,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "font.family": "serif",
        "font.serif": [
            "Computer Modern Roman",
            "CMU Serif",
            "Times New Roman",
            "DejaVu Serif",
        ],
        "mathtext.fontset": "cm",
        "mathtext.default": "it",
    }
)

# Import PyDMA modules
from pydma import (
    DMAAnalyzer,
    DMAConfig,
    ElectrodeOCP,
    BlendElectrode,
    load_ocp,
    plot_aging_study,
    plot_ocv_model_param_show,
    AgingStudyResults,
)

print("PyDMA imported successfully!")


## 3. Load Half-Cell Electrode Data

We load measured half-cell OCP curves from CSV files:
- **Si-Gr anode** (P45B, lithiation direction) 
- **NCA cathode** (GITT pOCP .csv)

In [ ]:
import pandas as pd
from pathlib import Path

# Define data paths
INPUT_DATA_DIR = Path("InputData")
if not INPUT_DATA_DIR.exists():
    INPUT_DATA_DIR = Path("TestData/InputData")
if not INPUT_DATA_DIR.exists():
    INPUT_DATA_DIR = Path("notebooks/TestData/InputData")

# Load Si-Gr anode OCP (P45B, lithiation direction for charging)
anode_df = pd.read_csv(INPUT_DATA_DIR / "SiC_blend_anode" / "P45B_Anode_Lithiation_0C02.csv")
print("Anode columns:", anode_df.columns.tolist())
print(f"Anode data: {len(anode_df)} points")

# Load NCA cathode OCP (MATLAB-aligned GITT pOCP .csv)
cathode_path = INPUT_DATA_DIR / "NCA" / "GITT_P45b_Cat_NCA_JN_VS_Coin_1_GITT__Extracted_Continuous_pOCP.csv"
cathode = load_ocp(cathode_path, electrode_type="cathode", smooth=False)
print(f"Cathode file: {cathode_path.name}")
print(f"Cathode data: {len(cathode.soc)} points")

# Create ElectrodeOCP objects
anode = ElectrodeOCP(
    soc=anode_df['normalizedCapacity'].values,
    voltage=anode_df['voltage'].values,
    name="Si-Gr Anode (P45B)"
)

# Plot the half-cell curves (separate panels)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

axes[0].plot(
    anode.soc,
    anode.voltage,
    '-',
    color=np.array([48, 112, 179]) / 255,
    linewidth=2,
    label='Anode (Si-Gr P45B)',
)
axes[0].set_xlabel('SOC / -')
axes[0].set_ylabel('$U$ / V')
axes[0].set_title('Anode OCP')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    cathode.soc,
    cathode.voltage,
    '-',
    color=np.array([162, 173, 0]) / 255,
    linewidth=2,
    label='Cathode (NCA)',
)
axes[1].set_xlabel('SOC / -')
axes[1].set_ylabel('$U$ / V')
axes[1].set_title('Cathode OCP')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAnode voltage range: {anode.voltage.min():.3f} - {anode.voltage.max():.3f} V")
print(f"Cathode voltage range: {cathode.voltage.min():.3f} - {cathode.voltage.max():.3f} V")


## 4. Load Full-Cell pOCV Data

Load the measured pseudo-OCV data from a P45B cell during charging.

In [ ]:
# Load full-cell pOCV measurement data
from pathlib import Path
import re

POCV_DIR = Path("TestData")
if not POCV_DIR.exists():
    POCV_DIR = Path("notebooks/TestData")

# Choose how many files to plot: "one", "three", or "all" (MATLAB comparison: use "all")
plot_mode = "three"
# Choose how many files to fit: "one", "three", or "all" (default: same as plot_mode)
fit_mode = plot_mode
analysis_entry = 1  # which entry to use for analysis when fit_mode="one"

# Find available files
files = sorted(POCV_DIR.glob("FR23_pOCV_CH_entry*.csv"))
if not files:
    raise FileNotFoundError("No FR23_pOCV_CH_entry*.csv files found in TestData")

# Helpers to extract entry identifiers
def _entry_token(path):
    m = re.search(r"entry(\d+)", path.name, re.IGNORECASE)
    return m.group(1) if m else None


def _entry_num(path):
    token = _entry_token(path)
    return int(token) if token is not None else None


files = sorted(files, key=lambda p: _entry_num(p) or 0)

# Helper to select files by mode
def _select_files(mode, file_list):
    if mode == "one":
        return [file_list[0]]
    if mode == "three":
        return file_list[:3]
    if mode == "all":
        return file_list
    raise ValueError("mode must be 'one', 'three', or 'all'")


# Select files to plot
plot_files = _select_files(plot_mode, files)

# Plot selected pOCV curves (smaller figure and legend labels as RPTxx)
plot_preview_df = pd.read_csv(plot_files[0])
title_battery_name = (
    str(plot_preview_df["Battery_name"].iloc[0])
    if "Battery_name" in plot_preview_df.columns
    else "P45B"
)

def _clean_serial(value):
    try:
        numeric = float(value)
        if numeric.is_integer():
            return str(int(numeric))
    except (TypeError, ValueError):
        pass
    return str(value)


if "Battery_serial" in plot_preview_df.columns:
    title_battery_serial = _clean_serial(plot_preview_df["Battery_serial"].iloc[0])
else:
    serial_match = re.search(r"FR(\d+)", plot_files[0].name, re.IGNORECASE)
    title_battery_serial = serial_match.group(1) if serial_match else "?"

plt.figure(figsize=(6, 3))
for f in plot_files:
    df = pd.read_csv(f)
    cap = df["Ah_Step"].values
    volt = df["U"].values
    mask = ~(np.isnan(cap) | np.isnan(volt))
    cap = cap[mask]
    volt = volt[mask]

    entry_token = _entry_token(f)
    label = f"RPT{entry_token}" if entry_token is not None else "RPT"
    plt.plot(cap, volt, linewidth=1.0, label=label)

plt.xlabel(r"$Q$ / $\mathrm{Ah}$", fontsize=9)
plt.ylabel(r"$U$ / $\mathrm{V}$", fontsize=9)
plt.title(
    f"{title_battery_name} Serial {title_battery_serial} Full-Cell Pseudo-OCV (Charging)",
    fontsize=8,
)
plt.legend(fontsize=7, frameon=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Select files to fit
if fit_mode == "one":
    analysis_file = None
    for f in files:
        if _entry_num(f) == analysis_entry:
            analysis_file = f
            break
    if analysis_file is None:
        analysis_file = files[0]
    fit_files = [analysis_file]
else:
    fit_files = _select_files(fit_mode, files)
    analysis_file = fit_files[0]

# Build fit data list: (cu_name, capacity, voltage, efc)
fit_data = []
for f in fit_files:
    df = pd.read_csv(f)
    cap = df["Ah_Step"].values
    volt = df["U"].values
    mask = ~(np.isnan(cap) | np.isnan(volt))
    cap = cap[mask]
    volt = volt[mask]
    efc = float(df["EFC"].iloc[0]) if "EFC" in df.columns else None
    cu_name = f"CU{_entry_num(f)}" if _entry_num(f) is not None else f.stem
    fit_data.append((cu_name, cap, volt, efc))

# Load the selected analysis entry (for single-CU prints)
meas_full_cell = pd.read_csv(analysis_file)
print("Full-cell data columns:", meas_full_cell.columns.tolist())
print(f"Using analysis file: {analysis_file.name}")

# Extract capacity and voltage
# Using Ah_Step for capacity and U for actual voltage measurement
meas_capacity = meas_full_cell['Ah_Step'].values
meas_voltage = meas_full_cell['U'].values  # U is the actual measured voltage

# Clean data - remove any NaN values
valid_mask = ~(np.isnan(meas_capacity) | np.isnan(meas_voltage))
meas_capacity = meas_capacity[valid_mask]
meas_voltage = meas_voltage[valid_mask]

# Get cell info
battery_name = meas_full_cell['Battery_name'].iloc[0]
battery_serial = meas_full_cell['Battery_serial'].iloc[0]
print(f"\nBattery: {battery_name} Serial: {battery_serial}")
print(f"Data points: {len(meas_capacity)}")
print(f"Capacity range: {meas_capacity.min():.4f} - {meas_capacity.max():.4f} Ah")
print(f"Voltage range: {meas_voltage.min():.3f} - {meas_voltage.max():.3f} V")

# Reference capacity (total capacity of the measurement)
reference_capacity = meas_capacity.max() - meas_capacity.min()
print(f"Reference capacity: {reference_capacity:.4f} Ah")


## 5. Configure the DMA Analyzer

Set up the analysis configuration. The `speed_preset` parameter controls the trade-off between accuracy and computation time.

In [ ]:
# Create configuration
# NOTE: For quick tutorial runs, some settings differ from MATLAB defaults.
# See comments for MATLAB-equivalent values.
accepted_solutions_target = 2  # exact target for robust MATLAB comparison

config = DMAConfig(
    # 'fast', 'medium', or 'thorough' (MATLAB-equivalent)
    # - fast: quick tests (~3% of MATLAB compute)
    # - medium: balanced (~30% of MATLAB compute)
    # - thorough: MATLAB-equivalent (popsize=500, maxiter=100)
    speed_preset='fast',
    direction='charge',
    data_length=1000,             # MATLAB default: 1000 ✓
    smoothing_points=30,          # MATLAB default: 30 ✓
    # ROI bounds are SOC/Q based (MATLAB-style), not voltage bounds
    roi_ocv_min=0.0,              # MATLAB default: 0 ✓
    roi_ocv_max=1.0,              # MATLAB default: 1 ✓
    roi_dva_min=0.10,             # MATLAB default: 0.1 ✓
    roi_dva_max=0.90,             # MATLAB default: 0.9 ✓
    roi_ica_min=0.13,             # MATLAB default: 0.13 ✓
    roi_ica_max=0.90,             # MATLAB default: 0.9 ✓
    # Cost function weights (MATLAB defaults)
    weight_ocv=100.0,             # MATLAB default: 100 ✓
    weight_dva=1.0,               # MATLAB default: 1 ✓
    weight_ica=0.0,               # MATLAB default: 0 ✓
    # Robust optimization for MATLAB-style comparison runs
    req_accepted=accepted_solutions_target,  # requested: >=2 and <=5
    max_tries_overall=5,          # Set to 5 tries for this tutorial
    # Acceptance threshold is OCV RMSE in volts (displayed in mV)
    rmse_threshold=0.01,          # MATLAB default: 0.01 (10 mV)
    
    # ============================================================
    # INHOMOGENEITY SETTINGS (models non-uniform SOC distribution)
    # ============================================================
    # MATLAB Reference: main_DMA.m lines 214-221
    # MATLAB defaults: allowAnodeInhomogeneity=true, allowCathodeInhomogeneity=true
    # Set to False here for faster tutorial; enable for production use
    # Do NOT use cathode inhomogeneity for LFP cells!
    allow_anode_inhomogeneity=False,      # MATLAB default: true
    allow_cathode_inhomogeneity=False,    # MATLAB default: true
    allow_first_cycle_inhomogeneity=True, # MATLAB default: true ✓
    max_inhomogeneity=0.3,                # MATLAB default: 0.3 ✓
    max_inhomogeneity_delta=0.1,          # MATLAB default: 0.1 ✓
    
    # ============================================================
    # PENALTY CONSTRAINTS (applied automatically from 2nd CU onwards)
    # ============================================================
    # MATLAB Reference: main_DMA.m lines 225-234
    # These soft constraints penalize physically implausible degradation:
    # - "gain" = maximum allowed capacity regeneration per CU (LAM decrease)
    # - "loss" = maximum allowed degradation per CU (LAM increase)
    
    # Maximum allowed capacity regeneration (negative LAM change)
    max_anode_gain=0.01,          # MATLAB default: 0.01 ✓
    max_cathode_gain=0.01,        # MATLAB default: 0.01 ✓
    max_anode_blend1_gain=0.005,  # MATLAB default: 0.005 ✓
    max_anode_blend2_gain=0.005,   # MATLAB default: 0.01 ✓
    
    # Maximum allowed degradation per CU (1.0 = 100% = unlimited)
    max_anode_loss=1.0,           # MATLAB default: 1.0 ✓
    max_cathode_loss=1.0,         # MATLAB default: 1.0 ✓
    max_anode_blend1_loss=1.0,    # MATLAB default: 1.0 ✓
    max_anode_blend2_loss=1.0,    # MATLAB default: 1.0 ✓
)

# ============================================================
# SPECIAL NOTE FOR LFP CELLS:
# ============================================================
# For LFP (Lithium Iron Phosphate) cathodes, we recommend constraining
# alpha + beta ≈ 1.0 to improve fitting stability on the flat voltage plateau.
# You can enforce this using bounds parameter in DMAConfig:
#   bounds = (alpha_an_lower, beta_an_lower, alpha_ca_lower, beta_ca_lower,
#             alpha_an_upper, beta_an_upper, alpha_ca_upper, beta_ca_upper)
# Example for LFP: bounds = (0.9, -0.2, 0.9, 0.0, 1.1, 0.2, 1.0, 0.1)
# where alpha_ca_upper=1.0 and beta_ca_upper=0.1 ensures alpha+beta ≈ 1.0

# Print configuration summary
print("Configuration created:")
print(f"  Speed preset: {config.speed_preset}")
print(f"  Direction: {config.direction}")
print(f"  Data length: {config.data_length}")
print(f"  SOC ROI (DVA): [{config.roi_dva_min}, {config.roi_dva_max}]")
print(f"  Weights: OCV={config.weight_ocv}, DVA={config.weight_dva}, ICA={config.weight_ica}")
print(f"  Acceptance RMSE threshold: {config.rmse_threshold*1000:.1f} mV")
print(f"\nInhomogeneity settings:")
print(f"  Anode inhomogeneity: {config.allow_anode_inhomogeneity}")
print(f"  Cathode inhomogeneity: {config.allow_cathode_inhomogeneity}")
print(f"  Max inhomogeneity: {config.max_inhomogeneity*100:.0f}%")
print(f"\nPenalty constraints (for multi-CU fitting):")
print(f"  Max anode gain: {config.max_anode_gain*100:.1f}% per CU")
print(f"  Max cathode gain: {config.max_cathode_gain*100:.1f}% per CU")
print(f"  Max anode loss: {config.max_anode_loss*100:.0f}% per CU (unlimited)")


## 6. Run the Analysis

Create the analyzer and run the degradation mode analysis.

In [ ]:
import time

# Create analyzer
analyzer = DMAAnalyzer(config)

# Set electrode models
analyzer.set_anode(anode)
analyzer.set_cathode(cathode)

# Set reference capacity
analyzer.set_reference_capacity(reference_capacity)

# Progress callback (optional)
def progress_callback(accepted, rejected, total):
    print(f"  Run {total}: {accepted} accepted, {rejected} rejected")

# Toggle runtime measurement (MATLAB: s.measureRuntimePerCU)
measure_runtime = True

if fit_mode == "one":
    # Clear study_results from previous runs
    if 'study_results' in globals():
        del study_results
    
    print("Running DMA analysis...")
    print(f"  Anode: {anode.name}")
    print(f"  Cathode: {cathode.name}")
    print(f"  Reference capacity: {reference_capacity:.4f} Ah")
    print("(This may take a few seconds)\n")

    # Run analysis with optional timing
    if measure_runtime:
        t_start = time.perf_counter()
    
    result = analyzer.analyze(
        measured_capacity=meas_capacity,
        measured_voltage=meas_voltage,
        progress_callback=progress_callback,
    )
    
    if measure_runtime:
        t_elapsed = time.perf_counter() - t_start
        print(f"  Runtime: {t_elapsed:.2f} s")
else:
    print(f"Running DMA analysis for {len(fit_data)} CUs...")
    print(f"  Anode: {anode.name}")
    print(f"  Cathode: {cathode.name}")
    print("(This may take a few seconds per CU)\n")

    # Build pocv_data dict and efc_values for analyze_aging_study
    pocv_data = {cu_name: (cap, volt) for cu_name, cap, volt, efc in fit_data}
    efc_values = {cu_name: efc for cu_name, cap, volt, efc in fit_data if efc is not None}

    if measure_runtime:
        t_start = time.perf_counter()

    study_results = analyzer.analyze_aging_study(
        pocv_data=pocv_data,
        efc_values=efc_values or None,
        progress_callback=progress_callback,
    )

    if measure_runtime:
        t_elapsed = time.perf_counter() - t_start
        print(f"  Total runtime: {t_elapsed:.2f} s")

    # Use last CU result for downstream single-CU plots
    result = study_results.results[study_results.cu_labels[-1]]

print("-- Analysis complete!")

## 7. Examine the Results

In [ ]:
# Print results summary
# Always use the same summary-style output (single-CU or multi-CU)
if 'study_results' in globals() and len(study_results) > 0:
    results_std = study_results
else:
    results_std = AgingStudyResults()
    r0 = result
    if not getattr(r0, "cu_name", ""):
        r0.cu_name = "CU1"
    results_std.add_result(r0, efc=None)

print(f"\nFITTED PARAMETERS ({len(results_std)} Check-Ups):")
print("-" * 70)
# Use built-in PyDMA summary for fitted parameters (standard)
for cu_name in results_std.cu_labels:
    r = results_std.results[cu_name]
    print(f"{cu_name}:")
    print(r.fitted_params)
print("-" * 70)

print("\nDEGRADATION MODES:")
if results_std.cu_labels:
    # Use built-in PyDMA summary for degradation modes (standard)
    for cu_name in results_std.cu_labels:
        r = results_std.results[cu_name]
        print(f"{cu_name}:")
        print(r.degradation_modes)
else:
    print("  (no degradation modes to show)")

# Show EFC values if available
if results_std.efc_values:
    print("\nEFC VALUES:")
    print("-" * 70)
    for cu_name, efc in zip(results_std.cu_labels, results_std.efc_values):
        print(f"  {cu_name}: EFC = {efc:.1f}")


## 8. Visualize the Results

**Note:** The DVA panel is plotted over SOC (0-1) and uses normalized scaling (dU/dQ_normalized).

In [ ]:
# ============================================================
# VISUALIZATION SETTINGS
# ============================================================
# Number of CUs to show OCV/DVA plots, driven by global `plot_mode`
#   plot_mode='one'   -> last 1 CU
#   plot_mode='three' -> last 3 CUs (or fewer if unavailable)
#   plot_mode='all'   -> all fitted CUs
if plot_mode == "one":
    n_cus_to_plot = 1
elif plot_mode == "three":
    n_cus_to_plot = min(3, len(fit_data))
elif plot_mode == "all":
    n_cus_to_plot = len(fit_data)
else:
    raise ValueError("plot_mode must be 'one', 'three', or 'all'")

import re


def _cu_index(name):
    """Extract numeric CU index from strings like 'CU3' or 'entry03'."""
    if not name:
        return None
    match = re.search(r"(\d+)", str(name))
    return int(match.group(1)) if match else None


def plot_ocv_dva_matlab_style(
    result,
    *,
    cell_name=None,
    cu_name=None,
    label_cfg=None,
    figsize_cm=(13, 12),
):
    """Plot MATLAB-style OCV + DVA with alpha/beta arrows."""
    return plot_ocv_model_param_show(
        result,
        cell_name=cell_name,
        cu_index=_cu_index(cu_name if cu_name else getattr(result, "cu_name", "")),
        label_cfg=label_cfg,
        figsize_cm=figsize_cm,
        legend_ncols=2,
        latex_fonts=True,
    )


cell_title_name = battery_name if "battery_name" in globals() else None
default_labels = {
    "label_cathode": "Cathode",
    "label_anode": "Anode",
}

# Plot OCV/DVA for requested number of CUs (MATLAB-style figure)
if fit_mode == "one" or "study_results" not in globals():
    cu_name = result.cu_name if hasattr(result, "cu_name") and result.cu_name else "CU1"
    fig = plot_ocv_dva_matlab_style(
        result,
        cell_name=cell_title_name,
        cu_name=cu_name,
        label_cfg=default_labels,
    )
    plt.show()
else:
    # Multi-CU mode: plot last n_cus_to_plot CUs
    cu_labels_to_plot = study_results.cu_labels[-n_cus_to_plot:]

    for cu_name in cu_labels_to_plot:
        res = study_results.results[cu_name]
        fig = plot_ocv_dva_matlab_style(
            res,
            cell_name=cell_title_name,
            cu_name=cu_name,
            label_cfg=default_labels,
        )
        plt.show()

# Aging study plot (if multiple CUs were fitted)
# MATLAB-style: separate panels for Cathode, Anode, LLI, RMSE - all in TUM Blue
if fit_mode != "one" and "study_results" in globals():
    fig_age = plot_aging_study(
        study_results,
        x_label="EFC" if study_results.efc_values else "CU",
        plot_cathode=True,   # Show/hide cathode LAM panel
        plot_anode=True,     # Show/hide anode LAM panel
        plot_lli=True,       # Show/hide LLI panel
        plot_rmse=True,      # Show/hide RMSE panel
        calendar_aging=False,  # True for calendar aging (shows "RPT Number")
    )
    plt.show()

## 9. Working with Blend Electrodes (Si-Gr)

PyDMA supports blend electrode models, commonly used for silicon-graphite anodes.

In [ ]:
from pathlib import Path
if "INPUT_DATA_DIR" not in globals():
    INPUT_DATA_DIR = Path("TestData/InputData")
    if not INPUT_DATA_DIR.exists():
        INPUT_DATA_DIR = Path("notebooks/TestData/InputData")

# Load graphite OCP data for blend component 1
graphite_df = pd.read_csv(INPUT_DATA_DIR / "Graphite" / "Gr_Lithiation_Kuecher.csv")
print(f"Graphite data: {len(graphite_df)} points")

# Create graphite electrode object
graphite = ElectrodeOCP(
    soc=graphite_df['normalizedCapacity'].values,
    voltage=graphite_df['voltage'].values,
    name="Graphite (Kuecher)"
)

# Load silicon OCP data (reconstructed curve from P45B anode)
silicon_df = pd.read_csv(INPUT_DATA_DIR / "Silicon" / "SiReconstr_Lithiation_Kuecher_P45B_Anode_0C03.csv")
print(f"Silicon data: {len(silicon_df)} points")

# Create silicon electrode object
silicon = ElectrodeOCP(
    soc=silicon_df['normalizedCapacity'].values,
    voltage=silicon_df['voltage'].values,
    name="Silicon (Reconstructed)"
)

# Create blend electrode (Si-Gr anode)
# blend1 = primary component (graphite), blend2 = secondary (silicon)
si_gr_anode = BlendElectrode(
    blend1=graphite,  # Graphite as primary
    blend2=silicon,  # Silicon as secondary
    name="Si-Gr Anode (P45B)"
)

# Plot blend components
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(graphite.soc, graphite.voltage, 'b-', linewidth=2, label='Graphite (Kuecher)')
ax.plot(silicon.soc, silicon.voltage, 'g-', linewidth=2, label='Silicon (Reconstructed)')

# Plot blends with different gamma values
# P45B has ~24.5% silicon content
for gamma in [0.10, 0.245, 0.35]:
    soc_blend, v_blend = si_gr_anode.get_blend_curve(gamma)
    label = f'Blend (γ={gamma})' if gamma != 0.245 else f'Blend (γ={gamma}) - P45B typical'
    ax.plot(soc_blend, v_blend, '--', linewidth=1.5, label=label)

ax.set_xlabel('SOC / -')
ax.set_ylabel('$U$ / V')
ax.set_title('Si-Gr blend electrode at different gamma values', fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("\nNote: γ (gamma) represents the silicon fraction in the blend.")
print("Q_blend = γ·Q_Si + (1-γ)·Q_Gr")


## 9b. Run DMA Analysis with Si-Gr Blend Anode

Now let's run the actual degradation mode analysis using the Si-Gr blend anode instead of pure graphite. This is the proper way to analyze P45B cells since they contain ~24.5% silicon in the anode.

In [ ]:
# Configure for blend electrode analysis
# NOTE: Settings aligned with MATLAB defaults for best DVA fit quality
config_blend = DMAConfig(
    speed_preset='fast',            # 'fast', 'medium', or 'thorough' (MATLAB-equivalent)
    workers=-1,                       # Use all CPU cores for parallel optimization
    direction='charge',
    data_length=1000,                 # MATLAB default: 1000 ✓
    smoothing_points=30,              # MATLAB default: 30 ✓
    # ROI bounds are SOC/Q based (MATLAB-style)
    roi_ocv_min=0.0,                  # MATLAB default: 0 ✓
    roi_ocv_max=1.0,                  # MATLAB default: 1 ✓
    roi_dva_min=0.10,                 # MATLAB default: 0.1 ✓
    roi_dva_max=0.90,                 # MATLAB default: 0.9 ✓
    roi_ica_min=0.13,                 # MATLAB default: 0.13 ✓
    roi_ica_max=0.90,                 # MATLAB default: 0.9 ✓
    # Cost function weights (MATLAB defaults)
    weight_ocv=100.0,                 # MATLAB default: 100 ✓
    weight_dva=1.0,                   # MATLAB default: 1 ✓
    weight_ica=0.0,                   # MATLAB default: 0 ✓
    
    # Enable blend fitting for anode (MATLAB default for P45B)
    use_anode_blend=True,             # MATLAB default: true ✓
    use_cathode_blend=False,          # MATLAB default: false ✓
    gamma_anode_blend2_init=0.25,     # MATLAB default: 0.25 ✓
    gamma_anode_blend2_upper=0.30,    # MATLAB default: 0.30 ✓
    
    # Robust acceptance criteria for comparison-quality runs
    req_accepted=accepted_solutions_target,  # requested: >=2 and <=5
    max_tries_overall=5,              # Set to 5 tries for this tutorial
    rmse_threshold=0.01,              # 10 mV (MATLAB: 10 mV)
    
    # ============================================================
    # INHOMOGENEITY SETTINGS (models non-uniform SOC distribution)
    # ============================================================
    # MATLAB defaults: allowAnodeInhomogeneity=true, allowCathodeInhomogeneity=true
    # Disabled for tutorial; enable for production use
    allow_anode_inhomogeneity=True,      # MATLAB default: true; disabled for speed
    allow_cathode_inhomogeneity=True,    # MATLAB default: true; disabled for speed
    allow_first_cycle_inhomogeneity=True, # MATLAB default: true ✓
    max_inhomogeneity=0.3,                # MATLAB default: 0.3 ✓
    max_inhomogeneity_delta=0.1,          # MATLAB default: 0.1 ✓
    
    # ============================================================
    # PENALTY CONSTRAINTS (applied automatically from 2nd CU onwards)
    # ============================================================
    # For blend analysis, these also constrain Graphite/Silicon separately
    max_anode_gain=0.01,              # MATLAB default: 0.01 ✓
    max_cathode_gain=0.01,            # MATLAB default: 0.01 ✓
    max_anode_blend1_gain=0.005,      # MATLAB default: 0.005 ✓
    max_anode_blend2_gain=0.01,       # MATLAB default: 0.01 ✓
    max_anode_loss=1.0,               # MATLAB default: 1.0 ✓
    max_cathode_loss=1.0,             # MATLAB default: 1.0 ✓
    max_anode_blend1_loss=1.0,        # MATLAB default: 1.0 ✓
    max_anode_blend2_loss=1.0,        # MATLAB default: 1.0 ✓
)

# Create analyzer with blend anode
analyzer_blend = DMAAnalyzer(config_blend)
analyzer_blend.set_anode(si_gr_anode)  # Use the Si-Gr blend electrode!
analyzer_blend.set_cathode(cathode)
analyzer_blend.set_reference_capacity(reference_capacity)

if fit_mode == "one":
    # Clear study_results_blend from previous runs
    if 'study_results_blend' in globals():
        del study_results_blend
    
    print("Running DMA analysis with Si-Gr BLEND anode...")
    print(f"  Anode: {si_gr_anode.name} (BLEND)")
    print(f"  Cathode: {cathode.name}")
    print(f"  Blend enabled: gamma_Si will be fitted")
    print(f"  Inhomogeneity: anode={config_blend.allow_anode_inhomogeneity}, cathode={config_blend.allow_cathode_inhomogeneity}")
    print(f"  Parallelization: workers={config_blend.workers}")
    print("(This may take longer due to additional parameters)\n")

    # Run analysis with optional timing
    if measure_runtime:
        t_start = time.perf_counter()
    
    result_blend = analyzer_blend.analyze(
        measured_capacity=meas_capacity,
        measured_voltage=meas_voltage,
        progress_callback=progress_callback,
    )
    
    if measure_runtime:
        t_elapsed = time.perf_counter() - t_start
        print(f"  Runtime: {t_elapsed:.2f} s")
else:
    print(f"Running BLEND DMA analysis for {len(fit_data)} CUs...")
    print(f"  Anode: {si_gr_anode.name} (BLEND)")
    print(f"  Cathode: {cathode.name}")
    print(f"  Blend enabled: gamma_Si will be fitted")
    print(f"  Inhomogeneity: anode={config_blend.allow_anode_inhomogeneity}, cathode={config_blend.allow_cathode_inhomogeneity}")
    print(f"  Parallelization: workers={config_blend.workers}")
    print(f"  Speed preset: {config_blend.speed_preset}")
    print(f"  Acceptance: {config_blend.req_accepted} runs with RMSE < {config_blend.rmse_threshold*1000:.0f} mV")
    print("(Use 'thorough' preset for production/publication)\n")

    # Build pocv_data dict and efc_values for analyze_aging_study
    pocv_data_blend = {cu_name: (cap, volt) for cu_name, cap, volt, efc in fit_data}
    efc_values_blend = {cu_name: efc for cu_name, cap, volt, efc in fit_data if efc is not None}

    if measure_runtime:
        t_start = time.perf_counter()

    study_results_blend = analyzer_blend.analyze_aging_study(
        pocv_data=pocv_data_blend,
        efc_values=efc_values_blend or None,
        progress_callback=progress_callback,
    )

    if measure_runtime:
        t_elapsed = time.perf_counter() - t_start
        print(f"  Total runtime: {t_elapsed:.2f} s")

    # Use last CU result for downstream single-CU plots
    result_blend = study_results_blend.results[study_results_blend.cu_labels[-1]]

print("BLEND Analysis complete!")

In [ ]:
# Display blend analysis results
# Always use the same summary-style output (single-CU or multi-CU)
if 'study_results_blend' in globals() and len(study_results_blend) > 0:
    results_blend = study_results_blend
else:
    results_blend = AgingStudyResults()
    rb = result_blend
    if not getattr(rb, "cu_name", ""):
        rb.cu_name = "CU1"
    results_blend.add_result(rb, efc=None)

print(f"\nFITTED PARAMETERS ({len(results_blend)} Check-Ups):")
print("-" * 100)
# Use built-in PyDMA summary for fitted parameters (blend)
for cu_name in results_blend.cu_labels:
    r = results_blend.results[cu_name]
    print(f"{cu_name}:")
    print(r.fitted_params)
print("-" * 100)

print("\nDEGRADATION MODES:")
if results_blend.cu_labels:
    # Use built-in PyDMA summary for degradation modes (blend)
    for cu_name in results_blend.cu_labels:
        r = results_blend.results[cu_name]
        print(f"{cu_name}:")
        print(r.degradation_modes)
else:
    print("  (no degradation modes to show)")

# Show EFC values if available
if results_blend.efc_values:
    print("\nEFC VALUES:")
    print("-" * 100)
    for cu_name, efc in zip(results_blend.cu_labels, results_blend.efc_values):
        print(f"  {cu_name}: EFC = {efc:.1f}")


In [ ]:
# ============================================================
# BLEND VISUALIZATION SETTINGS
# ============================================================
# Number of CUs to show OCV/DVA plots, driven by global `plot_mode`
#   plot_mode='one'   -> last 1 CU
#   plot_mode='three' -> last 3 CUs (or fewer if unavailable)
#   plot_mode='all'   -> all fitted CUs
if plot_mode == "one":
    n_cus_to_plot_blend = 1
elif plot_mode == "three":
    n_cus_to_plot_blend = min(3, len(fit_data))
elif plot_mode == "all":
    n_cus_to_plot_blend = len(fit_data)
else:
    raise ValueError("plot_mode must be 'one', 'three', or 'all'")

blend_labels = {
    "label_cathode": "Cathode",
    "label_anode": "Si-Gr",
}

cell_title_name = battery_name if "battery_name" in globals() else None

# Plot OCV/DVA for requested number of CUs (blend analysis, MATLAB-style)
if fit_mode == "one" or "study_results_blend" not in globals():
    cu_name = (
        result_blend.cu_name
        if hasattr(result_blend, "cu_name") and result_blend.cu_name
        else "CU1"
    )
    fig = plot_ocv_dva_matlab_style(
        result_blend,
        cell_name=cell_title_name,
        cu_name=cu_name,
        label_cfg=blend_labels,
    )
    plt.show()
else:
    # Multi-CU mode: plot last n_cus_to_plot_blend CUs
    cu_labels_to_plot = study_results_blend.cu_labels[-n_cus_to_plot_blend:]

    for cu_name in cu_labels_to_plot:
        res = study_results_blend.results[cu_name]
        fig = plot_ocv_dva_matlab_style(
            res,
            cell_name=cell_title_name,
            cu_name=cu_name,
            label_cfg=blend_labels,
        )
        plt.show()

# Aging study plot (if multiple CUs were fitted)
# MATLAB-style: separate panels for Cathode, Anode blend components, LLI, RMSE
if fit_mode != "one" and "study_results_blend" in globals():
    fig_blend_aging = plot_aging_study(
        study_results_blend,
        x_label="EFC" if study_results_blend.efc_values else "CU",
        plot_cathode=True,        # Show cathode LAM panel
        plot_anode=True,          # Show aggregate anode LAM panel
        plot_lli=True,            # Show LLI panel
        plot_rmse=True,            # Show RMSE panel
        use_anode_blend=True,     # Show separate Graphite/Silicon panels (MATLAB-style)
        calendar_aging=False,     # True for "RPT Number", False for "EFC"
        labels={
            "cathode": "Cathode",
            "anode": "Anode",
            "anode_blend1": "Graphite",    # Primary blend component
            "anode_blend2": "Silicon",      # Secondary blend component
            "lli": "Charge-carrier-inv",
            "rmse": "RMSE",
        },
    )
    plt.show()


## 10. Loading Real Electrode Data

In practice, you would load electrode data from files:

In [ ]:
# You can also use the load_ocp function for automatic column detection
from pydma import load_ocp

# Example: Load electrode data using auto-detection
cathode_loaded = load_ocp(INPUT_DATA_DIR / "NCA" / "GITT_P45b_Cat_NCA_JN_VS_Coin_1_GITT__Extracted_Continuous_pOCP.csv", electrode_type="cathode")
print(f"Loaded cathode: {cathode_loaded.name}")
print(f"  SOC range: {cathode_loaded.soc.min():.3f} - {cathode_loaded.soc.max():.3f}")
print(f"  Voltage range: {cathode_loaded.voltage.min():.3f} - {cathode_loaded.voltage.max():.3f} V")

# The function auto-detects column names like:
# - SOC: 'soc', 'normalizedCapacity', 'normalized_capacity', 'x', 'stoichiometry'
# - Voltage: 'voltage', 'ocp', 'potential', 'v', 'u'

print("\nAvailable electrode data in TestData folder:")
print("  Graphite: Gr_Lithiation_*.csv, Gr_Delithiation_*.csv")
print("  NCA: GITT_P45b_Cat_NCA_JN_VS_Coin_1_GITT__Extracted_Continuous_pOCP.csv, P45B_Cathode_*.csv, MoliM35A_PE_Cathode_*.csv")
print("  Silicon: Si_Lithiation_*.csv, SiReconstr_*.csv")


## 11. Tips for Best Results

1. **High-quality half-cell data**: The accuracy of DMA depends strongly on the quality of the half-cell OCP curves.

2. **Speed presets**: Use `speed_preset='thorough'` for publication-quality results (matches MATLAB). Use `'medium'` for a good balance of speed and accuracy (~30% of MATLAB compute time). Use `'fast'` only for quick testing.

3. **ROI selection**: Use SOC/Q-based bounds (`roi_ocv_min/max`, `roi_dva_min/max`, `roi_ica_min/max`).

4. **Multiple runs**: Use `req_accepted >= 3` to ensure robust results.

5. **Smoothing**: For noisy data, enable filtering in the configuration.

6. **Reference capacity**: Always set the reference capacity to the fresh cell capacity for proper degradation quantification.

## 12. Exporting Results

In [ ]:
# Export results to dictionary
result_dict = result.to_dict()
print("Result keys:", list(result_dict.keys()))

# Print key results
print(f"\n=== Summary for {battery_name} Serial {battery_serial} ===")
print(f"Degradation Modes:")
print(f"  LLI:    {result.degradation_modes.lli*100:+.2f}%")
print(f"  LAM_an: {result.degradation_modes.lam_an*100:+.2f}%")
print(f"  LAM_ca: {result.degradation_modes.lam_ca*100:+.2f}%")

# Save to JSON (example)
import json

# Note: numpy arrays need to be converted for JSON serialization
def convert_arrays(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_arrays(v) for k, v in obj.items()}
    else:
        return obj

# Save results to file
output_file = f"dma_results_{battery_name}_{battery_serial}.json"
with open(output_file, 'w') as f:
    json.dump(convert_arrays(result_dict), f, indent=2)
print(f"\nResults saved to: {output_file}")


## Summary

In this tutorial, you learned how to:

1. **Load real electrode OCP data** from CSV files (Graphite, NCA)
2. **Load full-cell pOCV measurement data** for analysis
3. **Configure the DMA analyzer** with appropriate parameters
4. **Run degradation mode analysis** to quantify LLI and LAM
5. **Visualize results** using built-in plotting functions
6. **Work with blend electrodes** (Si-Gr) for advanced analysis
7. **Export results** for further processing

### Next Steps

- Try analyzing multiple pOCV files (entry01 through entry09) to track aging
- Use `speed_preset='medium'` for balanced speed/accuracy, or `'thorough'` for publication-quality results
- Experiment with blend electrodes for Si-containing anodes
- Compare different electrode sources (Kuecher, Hossain, Wetjen, etc.)

For more information, see the [PyDMA documentation](https://github.com/TUM-EES/pydma).